# 01 การรวบรวมข้อมูล (Data Collection)

**เป้าหมาย:** นำเข้าไฟล์ดิบที่ผู้ใช้จัดเตรียมไว้แล้ว (ไม่ได้ดาวน์โหลดใหม่จากอินเทอร์เน็ต) พร้อมบันทึกที่มา (provenance) และ SHA256 ของแต่ละไฟล์

**อินพุต:** `accidents_2564..2569_original.csv` จาก `source_dir` ใน `backend/config/data_pipeline.json` (ค่าเริ่มต้นอ้างอิงจากโฟลเดอร์ backend คือ `../../RoadAware_2rd/data/raw/accidents`) หรือ env `ROADAWARE_SOURCE_DIR`

**การแปลง:** ไม่แก้ไขข้อมูลใด ๆ คัดลอกแบบ byte-for-byte เท่านั้น ถ้ามี snapshot เดิมที่ hash ไม่ตรงกับต้นทาง หรือ manifest เดิมบันทึก hash ต่างจากไฟล์ปัจจุบัน จะหยุดทันที ไม่เขียนทับและไม่รับรองไฟล์ที่ถูกแก้ไข

**เอาต์พุต:** `data/raw/accidents/*.csv` และ `data/raw/accidents/source_manifest.json` (อ้างอิงจาก output root = โฟลเดอร์ backend หรือ `--output-dir`)

**การตรวจสอบ:** ไฟล์ครบ 6 ไฟล์, hash ต้นทาง = hash snapshot, จำนวน record จาก pandas = จำนวน record จากโมดูล `csv`, บันทึกเวอร์ชัน Python/แพ็กเกจ

**ข้อจำกัด:** ไม่ทราบวันที่ดาวน์โหลดต้นฉบับ (`unknown`) จึงบันทึกเฉพาะเวลานำเข้า/ตรวจสอบ; แหล่งข้อมูลครอบคลุมเฉพาะโครงข่ายถนนของกระทรวงคมนาคม ไม่ใช่ทุกถนน; ไม่แสดงข้อมูลรายแถว

**สัญญาสำหรับขั้นถัดไป:** Notebook 02/03 อ่านเฉพาะ snapshot ที่ hash ตรงกับ `source_manifest.json` (ตรวจซ้ำทุกครั้งก่อนใช้)

In [1]:
import os, sys, json, platform, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd

def find_backend_root():
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).resolve()
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / 'backend' / 'config' / 'data_pipeline.json').is_file():
            return c / 'backend'
        if c.name == 'backend' and (c / 'config' / 'data_pipeline.json').is_file():
            return c
    raise FileNotFoundError('backend/config/data_pipeline.json not found from ' + str(here) + '; open the repository root or backend/notebooks')

BACKEND = find_backend_root()
REPO = BACKEND.parent
CFG = json.loads((BACKEND / 'config' / 'data_pipeline.json').read_text('utf-8'))
OUT = Path(os.environ.get('ROADAWARE_OUTPUT_DIR') or (BACKEND / CFG['output_root'])).resolve()
if os.environ.get('ROADAWARE_REQUIRE_VENV') == '1':
    assert Path(sys.prefix).resolve() == (REPO / '.venv').resolve(), 'kernel is not the project .venv: ' + sys.executable
RAW, REP = OUT / 'data' / 'raw' / 'accidents', OUT / 'reports' / 'data'
for d in (RAW, REP):
    d.mkdir(parents=True, exist_ok=True)
SRC = Path(os.environ.get('ROADAWARE_SOURCE_DIR') or (BACKEND / CFG['source_dir'])).resolve()
print('backend root:', BACKEND)
print('output root :', OUT)
print('source dir  :', SRC)
print('python      :', sys.executable)

backend root: /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend
output root : /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend
source dir  : /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware_2rd/data/raw/accidents
python      : /Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/.venv/bin/python


In [2]:
# FUNCTIONS (notebook 01) - self-contained; tests execute this cell directly
import csv, hashlib, shutil
import pandas as pd

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def import_snapshot(src, dst):
    """Copy src->dst idempotently. Never overwrite (or bless) a snapshot whose hash differs."""
    src_hash = sha256_file(src)
    if dst.exists():
        if sha256_file(dst) != src_hash:
            raise RuntimeError(f'Existing snapshot {dst.name} differs from source (sha256); refusing to overwrite or bless it. '
                               'Inspect the local file or rerun with a fresh --output-dir.')
        return 'already_present_hash_match', src_hash
    tmp = dst.with_name(dst.name + '.tmp')
    shutil.copyfile(src, tmp)
    if sha256_file(tmp) != src_hash:
        tmp.unlink()
        raise RuntimeError(f'Copy hash mismatch for {src.name}')
    tmp.replace(dst)
    return 'copied_hash_match', src_hash

def check_prior_manifest(prior, current_hashes):
    """Refuse to continue if an earlier manifest recorded a different hash for the same file name."""
    if not prior:
        return []
    old = {f['file']: f['sha256'] for f in prior.get('files', [])}
    changed = sorted(n for n, h in current_hashes.items() if n in old and old[n] != h)
    if changed:
        raise RuntimeError('Source/snapshot hash differs from the existing source_manifest.json for: ' + ', '.join(changed)
                           + '. Refusing to re-bless changed raw data; use a fresh --output-dir and document the new source.')
    return sorted(old)

def read_raw(path, encodings):
    last = None
    for enc in encodings:
        try:
            return pd.read_csv(path, dtype=str, keep_default_na=False, encoding=enc), enc
        except UnicodeDecodeError as e:
            last = e
    raise last

def count_csv_records(path, encoding):
    """Data records (non-empty CSV records minus the header); multiline quoted fields count once."""
    with open(path, newline='', encoding=encoding) as f:
        return sum(1 for rec in csv.reader(f) if rec) - 1

In [3]:
missing = [str(SRC / f) for f in CFG['source_files'].values() if not (SRC / f).is_file()]
if missing:
    raise FileNotFoundError('Missing source files (set ROADAWARE_SOURCE_DIR, runner --source-dir, or config source_dir):\n' + '\n'.join(missing))
src_hashes = {fname: sha256_file(SRC / fname) for fname in CFG['source_files'].values()}
man_path = RAW / 'source_manifest.json'
prior = json.loads(man_path.read_text('utf-8')) if man_path.exists() else None
previously_recorded = check_prior_manifest(prior, src_hashes)
verified_at = dt.datetime.now(dt.timezone.utc).isoformat()
records = []
for be_year, fname in CFG['source_files'].items():
    status, h = import_snapshot(SRC / fname, RAW / fname)
    assert h == src_hashes[fname], fname
    df, enc = read_raw(RAW / fname, CFG['encodings_to_try'])
    n_csv = count_csv_records(RAW / fname, enc)
    if n_csv != len(df):
        raise RuntimeError(f'{fname}: pandas rows {len(df)} != csv records {n_csv}; source_row ordinals would be unreliable')
    records.append({'be_year': int(be_year), 'ce_year': int(be_year) - 543, 'file': fname,
                    'original_path': str(SRC / fname), 'path': f'data/raw/accidents/{fname}',
                    'sha256': h, 'bytes': (RAW / fname).stat().st_size, 'encoding': enc,
                    'rows': len(df), 'n_columns': df.shape[1], 'columns': list(df.columns), 'import_status': status})
inv = pd.DataFrame(records)
inv[['be_year', 'file', 'bytes', 'rows', 'n_columns', 'encoding', 'import_status', 'sha256']]

,be_year,file,bytes,rows,n_columns,encoding,import_status,sha256
0,2564,accidents_2564_original.csv,10379365,20062,38,utf-8-sig,already_present_hash_match,044f8ae0a918ec25139bef24257fe274fd17d9bfab2512...
1,2565,accidents_2565_original.csv,10567379,20566,38,utf-8-sig,already_present_hash_match,6f7c543a302906925e32fbf6b8da5d2b1920266489ae1c...
2,2566,accidents_2566_original.csv,12093622,23841,38,utf-8-sig,already_present_hash_match,4d6dbfa4b8e1f4dd476d81062a01c10a858b7acd174fef...
3,2567,accidents_2567_original.csv,11260816,23952,38,utf-8-sig,already_present_hash_match,2ca1e832a4fa8d29ef6adfe52623a9b447a7f0b12e61e6...
4,2568,accidents_2568_original.csv,11984469,23715,38,utf-8-sig,already_present_hash_match,b163fa91170fa7df76b8d71b4eb629ff7ca4ef220bea6c...
5,2569,accidents_2569_original.csv,8320566,15239,38,utf-8-sig,already_present_hash_match,e9e0388614ce46cf815c1796d599d48e029be20b8d8849...


In [4]:
import matplotlib, nbformat
versions = {'python': sys.version.split()[0], 'python_executable': sys.executable, 'platform': platform.platform(),
            'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__, 'nbformat': nbformat.__version__}
manifest = {'source_url': CFG['source_url'], 'source_scope_note': CFG['source_scope_note'],
            'original_download_date': CFG['original_download_date'],
            'first_imported_at_utc': (prior or {}).get('first_imported_at_utc', verified_at),
            'last_verified_at_utc': verified_at,
            'collection_method': 'import of user-provided local files (no network download)',
            'source_row_definition': '1-based CSV record ordinal counting the header as record 1 (first data record = 2); not a physical line number when quoted fields span lines',
            'environment': versions, 'files': records}
man_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), 'utf-8')
base_cols = records[0]['columns']
schema_diff = pd.DataFrame([{'be_year': r['be_year'], 'same_header_as_2564': r['columns'] == base_cols,
    'only_in_file': [c for c in r['columns'] if c not in base_cols],
    'missing_vs_2564': [c for c in base_cols if c not in r['columns']]} for r in records])
schema_diff.to_csv(REP / 'collection_schema_diff.csv', index=False)
print(json.dumps(versions, indent=1))
schema_diff

{
 "python": "3.9.6",
 "python_executable": "/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/.venv/bin/python",
 "platform": "macOS-27.0.1-arm64-arm-64bit",
 "pandas": "2.3.3",
 "numpy": "2.0.2",
 "matplotlib": "3.9.4",
 "nbformat": "5.10.4"
}


,be_year,same_header_as_2564,only_in_file,missing_vs_2564
0,2564,True,[],[]
1,2565,True,[],[]
2,2566,True,[],[]
3,2567,True,[],[]
4,2568,True,[],[]
5,2569,False,"[ก.ม., บริเวณที่เกิดเหตุ/ลักษณะทาง, บริเวณที่เ...","[ACC_CODE, KM, บริเวณที่เกิดเหตุ, ลักษณะการเกิ..."


In [5]:
assert len(records) == 6 == len(CFG['source_files']), 'expected six source files'
for r in records:
    assert sha256_file(RAW / r['file']) == sha256_file(Path(r['original_path'])) == r['sha256'], r['file']
    assert r['rows'] > 0, r['file']
assert manifest['original_download_date'] == 'unknown'
print('Collection checks passed: 6 files, snapshot hashes == source hashes, pandas/csv record counts agree.')
print('rows per file:', dict(zip(inv.be_year, inv.rows)), 'total =', int(inv.rows.sum()))
print('previously recorded files re-verified:', len(previously_recorded))

Collection checks passed: 6 files, snapshot hashes == source hashes, pandas/csv record counts agree.
rows per file: {2564: 20062, 2565: 20566, 2566: 23841, 2567: 23952, 2568: 23715, 2569: 15239} total = 127375
previously recorded files re-verified: 6


## สรุปและข้อจำกัด

- นี่คือการนำเข้าไฟล์ที่มีอยู่แล้ว ไม่ใช่การเก็บข้อมูลใหม่ วันที่ดาวน์โหลดต้นฉบับไม่ทราบ (`original_download_date = unknown`); `first_imported_at_utc` คือเวลานำเข้าครั้งแรกในโฟลเดอร์ output นี้
- `source_row` ที่ใช้ต่อไปคือ **ลำดับ record ของ CSV แบบเริ่มที่ 1 โดยนับ header เป็น record 1** (record ข้อมูลแรก = 2) ไม่จำเป็นต้องเท่ากับเลขบรรทัดจริงหากมีฟิลด์ที่ขึ้นบรรทัดใหม่ในเครื่องหมายคำพูด
- หาก snapshot ในเครื่องถูกแก้ไข notebook นี้และ notebook ถัดไปจะหยุดทำงาน แทนที่จะรับรองไฟล์ที่เปลี่ยน
- ขั้นถัดไป: Notebook 02 (EDA บนข้อมูลดิบ) และ 03 (ทำความสะอาด)